In [1]:
import cv2
import numpy as np
import torch
import math

In [2]:
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

Using cache found in C:\Users\HP/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2024-2-13 Python-3.9.13 torch-2.0.0+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients
Adding AutoShape... 


In [3]:
cap = cv2.VideoCapture('C:/Users/HP/Py Code/Neural Network/Pytorch/Object detection/v1.mp4')

In [4]:
#Tracker class to track the detected object based on the proximity between center points ob bboxes
class Tracker:
    def __init__(self):
        # Store the center positions of the objects
        self.center_points = {}
        # Keep the count of the IDs
        # each time a new object id detected, the count will increase by one
        self.id_count = 0


    def update(self, objects_rect):
        # Objects boxes and ids
        objects_bbs_ids = []

        # Get center point of new object
        for rect in objects_rect:
            x, y, w, h = rect
            cx = (x + x + w) // 2
            cy = (y + y + h) // 2

            # Find out if that object was detected already
            same_object_detected = False
            for id, pt in self.center_points.items():
                dist = math.hypot(cx - pt[0], cy - pt[1])

                if dist < 35:
                    self.center_points[id] = (cx, cy)
#                    print(self.center_points)
                    objects_bbs_ids.append([x, y, w, h, id])
                    same_object_detected = True
                    break

            # New object is detected we assign the ID to that object
            if same_object_detected is False:
                self.center_points[self.id_count] = (cx, cy)
                objects_bbs_ids.append([x, y, w, h, self.id_count])
                self.id_count += 1

        # Clean the dictionary by center points to remove IDS not used anymore
        new_center_points = {}
        for obj_bb_id in objects_bbs_ids:
            _, _, _, _, object_id = obj_bb_id
            center = self.center_points[object_id]
            new_center_points[object_id] = center

        # Update dictionary with IDs not used removed
        self.center_points = new_center_points.copy()
        return objects_bbs_ids

In [5]:

    
#object detection
def POINTS(event, x, y, flags, param):
    if event == cv2.EVENT_MOUSEMOVE :  
        colorsBGR = [x, y]
        print(colorsBGR)
        

cv2.namedWindow('FRAME')
cv2.setMouseCallback('FRAME', POINTS)

# Define the Tracker
tracker = Tracker()

# Define the region of interest polygon
area_1 = [(3, 2), (1000, 2), (1000, 500), (3, 500)]

area1 = set()     
    
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (1020, 500))

    # Draw the region of interest polygon
    cv2.polylines(frame, [np.array(area_1, np.int32)], True, (0, 255, 0), 3)

    results = model(frame)

    # Extract bounding boxes of detected 'person' objects
    list = []
    for index, row in results.pandas().xyxy[0].iterrows():
        x1, y1, x2, y2 = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
        b = str(row['name'])
        if 'person' in b:
            list.append([x1, y1, x2, y2])

    # Update tracker with detected bounding boxes
    boxes_ids = tracker.update(list)

    area1.clear()
    for box_id in boxes_ids:
        x, y, w, h, id = box_id
        cv2.rectangle(frame, (x, y), (w, h), (255, 0, 255), 2)
        cv2.putText(frame, str(id), (x, y), cv2.FONT_HERSHEY_PLAIN, 1, (255, 0, 0), 2)

        # Check if the object is within the defined region of interest
        result = cv2.pointPolygonTest(np.array(area_1, np.int32), (x,y), False)
        if result > 0:
            area1.add(id)

    p = len(area1)
    print(p)
    cv2.putText(frame, 'count:' + str(p), (20, 30), cv2.FONT_HERSHEY_PLAIN, 3, (0, 255, 0), 2)
    if p > 5:
        cv2.putText(frame, 'Overloaded', (20, 60), cv2.FONT_HERSHEY_PLAIN, 3, (0, 255, 0), 2)

    cv2.imshow('FRAME', frame)
    if cv2.waitKey(0) & 0xFF == 27: #press ESC key
        break

cap.release()
cv2.destroyAllWindows()
